# Challenge 12 Final Colab: focused stacking for Kaggle

Esta notebook final no vuelve a entrenar modelos base.

Esta pensada para tu situacion actual:

- `signal_features` es el modelo fuerte
- `knn_cleaning` aporta diversidad real
- `svm_preprocessing` es opcional

## Objetivo

Combinar los artefactos ya generados por las corridas base:

- `oof_probabilities.csv`
- `test_probabilities.csv`
- `summary.json` si existe

y buscar una combinacion final mas fuerte mediante:

1. `weighted average` con busqueda de pesos
2. ajuste de umbral (`threshold tuning`)
3. `LogisticRegression` como meta-modelo

## Diseno operativo

- pensado para una sesion limpia de Google Colab
- no requiere `training.csv`, `test.csv` ni `sample.csv`
- acepta multiples ZIPs de corridas previas
- tambien puede reanudarse si guardas su bundle final

## 0. Imports and global constants

In [1]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import gc
import itertools
import json
import platform
import re
import warnings
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import files  # type: ignore
else:
    files = None

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["savefig.bbox"] = "tight"

RANDOM_STATE = 301655
NOTEBOOK_SLUG = "challenge_12_final_stacking_colab"

## 1. Create the stacking workspace

In [2]:
if IN_COLAB:
    WORKSPACE_ROOT = Path("/content/challenge_final_stacking_workspace")
else:
    cwd = Path.cwd().resolve()
    candidate_roots = []
    for base in [cwd, *cwd.parents]:
        candidate_roots.extend(
            [
                base / "challenge" / "results-pre-stack",
                base / "results-pre-stack",
            ]
        )
    WORKSPACE_ROOT = next((path for path in candidate_roots if path.exists()), cwd / "challenge_final_stacking_workspace")

INPUT_BUNDLE_DIR = WORKSPACE_ROOT / "input_bundles"
OUTPUT_ROOT = WORKSPACE_ROOT / "output"
PERSIST_ROOT = OUTPUT_ROOT / NOTEBOOK_SLUG
CHECKPOINT_DIR = PERSIST_ROOT / "checkpoints"
SUBMISSION_DIR = WORKSPACE_ROOT / "submissions"
EXPORT_DIR = WORKSPACE_ROOT / "exports"
ARTIFACT_SEARCH_ROOT = WORKSPACE_ROOT

for path in [WORKSPACE_ROOT, INPUT_BUNDLE_DIR, OUTPUT_ROOT, PERSIST_ROOT, CHECKPOINT_DIR, SUBMISSION_DIR, EXPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("WORKSPACE_ROOT:", WORKSPACE_ROOT)
print("ARTIFACT_SEARCH_ROOT:", ARTIFACT_SEARCH_ROOT)
print("PERSIST_ROOT:", PERSIST_ROOT)

IN_COLAB: False
WORKSPACE_ROOT: /Users/williamfrankmonroymamani/Documents/mem/stadistics/mem-estadistics/challenge/results-pre-stack
ARTIFACT_SEARCH_ROOT: /Users/williamfrankmonroymamani/Documents/mem/stadistics/mem-estadistics/challenge/results-pre-stack
PERSIST_ROOT: /Users/williamfrankmonroymamani/Documents/mem/stadistics/mem-estadistics/challenge/results-pre-stack/output/challenge_12_final_stacking_colab


## 2. Inspect the workspace

In [3]:
print("Workspace directories:")
for path in [WORKSPACE_ROOT, INPUT_BUNDLE_DIR, OUTPUT_ROOT, PERSIST_ROOT, CHECKPOINT_DIR, SUBMISSION_DIR, EXPORT_DIR]:
    print("-", path)

existing_oof = sorted(path for path in ARTIFACT_SEARCH_ROOT.rglob("oof_probabilities.csv") if PERSIST_ROOT not in path.parents)
existing_test = sorted(path for path in ARTIFACT_SEARCH_ROOT.rglob("test_probabilities.csv") if PERSIST_ROOT not in path.parents)

print("\nAlready discovered files before upload:")
print("- oof_probabilities.csv:", len(existing_oof))
print("- test_probabilities.csv:", len(existing_test))

Workspace directories:
- /Users/williamfrankmonroymamani/Documents/mem/stadistics/mem-estadistics/challenge/results-pre-stack
- /Users/williamfrankmonroymamani/Documents/mem/stadistics/mem-estadistics/challenge/results-pre-stack/input_bundles
- /Users/williamfrankmonroymamani/Documents/mem/stadistics/mem-estadistics/challenge/results-pre-stack/output
- /Users/williamfrankmonroymamani/Documents/mem/stadistics/mem-estadistics/challenge/results-pre-stack/output/challenge_12_final_stacking_colab
- /Users/williamfrankmonroymamani/Documents/mem/stadistics/mem-estadistics/challenge/results-pre-stack/output/challenge_12_final_stacking_colab/checkpoints
- /Users/williamfrankmonroymamani/Documents/mem/stadistics/mem-estadistics/challenge/results-pre-stack/submissions
- /Users/williamfrankmonroymamani/Documents/mem/stadistics/mem-estadistics/challenge/results-pre-stack/exports

Already discovered files before upload:
- oof_probabilities.csv: 2
- test_probabilities.csv: 2


## 3. Optional: upload one or more model-result ZIP bundles

In [4]:
UPLOAD_RESULT_BUNDLES = False

if UPLOAD_RESULT_BUNDLES:
    if not IN_COLAB:
        raise RuntimeError("This upload helper is intended for Google Colab.")

    uploaded = files.upload()
    zip_names = [Path(name).name for name in uploaded if str(name).lower().endswith(".zip")]
    if not zip_names:
        raise ValueError("Upload one or more ZIP bundles.")

    for bundle_name in zip_names:
        bundle_path = INPUT_BUNDLE_DIR / bundle_name
        bundle_path.write_bytes(uploaded[bundle_name])
        with zipfile.ZipFile(bundle_path, "r") as zip_file:
            zip_file.extractall(WORKSPACE_ROOT)
        print("Restored bundle:", bundle_path)
else:
    print("Set UPLOAD_RESULT_BUNDLES = True to upload ZIP bundles produced by the base-model notebooks.")

Set UPLOAD_RESULT_BUNDLES = True to upload ZIP bundles produced by the base-model notebooks.


## 4. Helpers

In [5]:
DEFAULT_BUNDLE_NAME = "challenge_12_final_stacking_colab_resume.zip"


def save_current_figure(filename: str) -> Path:
    path = PERSIST_ROOT / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()
    return path


def write_json_atomic(path: Path, payload: dict | list) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    tmp_path.write_text(json.dumps(payload, indent=2))
    tmp_path.replace(path)


def save_dataframe_atomic(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)


def read_dataframe(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def create_resume_bundle(bundle_name: str = DEFAULT_BUNDLE_NAME) -> Path:
    bundle_path = EXPORT_DIR / bundle_name
    if bundle_path.exists():
        bundle_path.unlink()

    with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as zip_file:
        for root_path in [OUTPUT_ROOT, SUBMISSION_DIR]:
            if not root_path.exists():
                continue
            for nested in root_path.rglob("*"):
                if nested.is_dir():
                    continue
                relative_path = nested.relative_to(WORKSPACE_ROOT)
                zip_file.write(nested, arcname=str(relative_path))

    return bundle_path


def safe_slug(value: str) -> str:
    value = value.strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value or "model"


def make_unique_name(base_name: str, used: set[str]) -> str:
    candidate = safe_slug(base_name)
    if candidate not in used:
        used.add(candidate)
        return candidate
    index = 2
    while f"{candidate}_{index}" in used:
        index += 1
    unique = f"{candidate}_{index}"
    used.add(unique)
    return unique


def load_summary_if_present(path: Path) -> dict:
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text())
    except Exception:
        return {}


def discover_probability_artifacts() -> list[dict]:
    records = []
    used_names = set()
    for oof_path in sorted(ARTIFACT_SEARCH_ROOT.rglob("oof_probabilities.csv")):
        if PERSIST_ROOT in oof_path.parents:
            continue

        model_root = oof_path.parent
        test_path = model_root / "test_probabilities.csv"
        summary_path = model_root / "summary.json"
        if not test_path.exists():
            continue

        oof_df = pd.read_csv(oof_path).sort_values("id").reset_index(drop=True)
        test_df = pd.read_csv(test_path).sort_values("id").reset_index(drop=True)
        if not {"id", "prob_1", "y_true"}.issubset(oof_df.columns):
            continue
        if not {"id", "prob_1"}.issubset(test_df.columns):
            continue

        summary = load_summary_if_present(summary_path)
        if "source_model" in oof_df.columns and not oof_df["source_model"].empty:
            source_name = str(oof_df["source_model"].iloc[0])
        else:
            source_name = summary.get("model_key") or summary.get("notebook_slug") or model_root.name

        model_name = make_unique_name(source_name, used_names)
        records.append(
            {
                "model_name": model_name,
                "model_root": model_root,
                "summary_path": summary_path,
                "summary": summary,
                "oof_path": oof_path,
                "test_path": test_path,
                "oof_df": oof_df,
                "test_df": test_df,
            }
        )

    return records


def align_artifacts(records: list[dict]) -> tuple[pd.DataFrame, pd.DataFrame]:
    if len(records) < 2:
        raise RuntimeError("At least two base models are required for final stacking.")

    meta_oof_df = records[0]["oof_df"][["id", "y_true"]].copy()
    meta_test_df = records[0]["test_df"][["id"]].copy()

    for record in records:
        model_name = record["model_name"]
        oof_part = record["oof_df"][["id", "prob_1"]].rename(columns={"prob_1": model_name})
        test_part = record["test_df"][["id", "prob_1"]].rename(columns={"prob_1": model_name})
        meta_oof_df = meta_oof_df.merge(oof_part, on="id", how="inner")
        meta_test_df = meta_test_df.merge(test_part, on="id", how="inner")

    meta_oof_df = meta_oof_df.sort_values("id").reset_index(drop=True)
    meta_test_df = meta_test_df.sort_values("id").reset_index(drop=True)
    return meta_oof_df, meta_test_df


def base_model_summary_table(records: list[dict], meta_oof_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for record in records:
        model_name = record["model_name"]
        probs = meta_oof_df[model_name].to_numpy(dtype=np.float64)
        preds = (probs >= 0.5).astype(int)
        rows.append(
            {
                "model_name": model_name,
                "oof_accuracy": float(accuracy_score(meta_oof_df["y_true"], preds)),
                "positive_rate": float(preds.mean()),
                "oof_path": str(record["oof_path"]),
                "test_path": str(record["test_path"]),
                "summary_path": str(record["summary_path"]) if record["summary_path"].exists() else "",
                "model_name_raw": record["summary"].get("model_name", ""),
                "best_stage": record["summary"].get("best_stage", ""),
            }
        )
    return pd.DataFrame(rows).sort_values(["oof_accuracy", "model_name"], ascending=[False, True]).reset_index(drop=True)


def pairwise_disagreement_table(meta_df: pd.DataFrame, model_names: list[str]) -> pd.DataFrame:
    rows = []
    for left_name, right_name in itertools.combinations(model_names, 2):
        left_pred = (meta_df[left_name] >= 0.5).astype(int)
        right_pred = (meta_df[right_name] >= 0.5).astype(int)
        rows.append(
            {
                "left_model": left_name,
                "right_model": right_name,
                "disagreement_rate": float((left_pred != right_pred).mean()),
            }
        )
    return pd.DataFrame(rows).sort_values("disagreement_rate", ascending=False).reset_index(drop=True)


def generate_weight_vectors(size: int, step: float) -> list[list[float]]:
    grid = np.round(np.arange(0.0, 1.0 + step / 2.0, step), 6)
    if size == 2:
        return [[float(w), float(round(1.0 - w, 6))] for w in grid]
    if size == 3:
        vectors = []
        for w1 in grid:
            for w2 in grid:
                w3 = round(1.0 - float(w1) - float(w2), 6)
                if w3 < -1e-9 or w3 > 1.0 + 1e-9:
                    continue
                if abs(float(w1) + float(w2) + w3 - 1.0) > 1e-6:
                    continue
                vectors.append([float(w1), float(w2), float(w3)])
        return vectors
    raise ValueError("Only subset sizes 2 and 3 are supported for weighted averages.")


def find_best_threshold(y_true: np.ndarray, prob_1: np.ndarray, threshold_grid: np.ndarray) -> tuple[float, float]:
    best_threshold = 0.5
    best_accuracy = -1.0
    for threshold in threshold_grid:
        preds = (prob_1 >= float(threshold)).astype(int)
        accuracy = accuracy_score(y_true, preds)
        if accuracy > best_accuracy + 1e-12:
            best_accuracy = float(accuracy)
            best_threshold = float(threshold)
    return best_threshold, best_accuracy


def weighted_probabilities(meta_df: pd.DataFrame, subset: list[str], weights: list[float]) -> np.ndarray:
    matrix = meta_df[subset].to_numpy(dtype=np.float64)
    weight_array = np.asarray(weights, dtype=np.float64)
    return (matrix * weight_array[None, :]).sum(axis=1)


def evaluate_weighted_candidate(meta_df: pd.DataFrame, subset: list[str], weights: list[float], threshold_grid: np.ndarray) -> dict:
    prob_1 = weighted_probabilities(meta_df, subset, weights)
    threshold, accuracy = find_best_threshold(meta_df["y_true"].to_numpy(dtype=np.int8), prob_1, threshold_grid)
    return {
        "meta_family": "weighted_average",
        "base_models": subset,
        "weights": weights,
        "C": None,
        "best_threshold": threshold,
        "meta_oof_accuracy": accuracy,
    }


def generate_logreg_oof_probabilities(meta_df: pd.DataFrame, subset: list[str], c_value: float) -> np.ndarray:
    X_meta = meta_df[subset].to_numpy(dtype=np.float32)
    y_meta = meta_df["y_true"].to_numpy(dtype=np.int8)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    oof_prob_1 = np.zeros(len(meta_df), dtype=np.float32)
    for fit_idx, eval_idx in cv.split(X_meta, y_meta):
        model = LogisticRegression(C=float(c_value), max_iter=2000)
        model.fit(X_meta[fit_idx], y_meta[fit_idx])
        oof_prob_1[eval_idx] = model.predict_proba(X_meta[eval_idx])[:, 1].astype(np.float32)
    return oof_prob_1


def evaluate_logreg_candidate(meta_df: pd.DataFrame, subset: list[str], c_value: float, threshold_grid: np.ndarray) -> dict:
    prob_1 = generate_logreg_oof_probabilities(meta_df, subset, c_value)
    threshold, accuracy = find_best_threshold(meta_df["y_true"].to_numpy(dtype=np.int8), prob_1, threshold_grid)
    return {
        "meta_family": "logreg",
        "base_models": subset,
        "weights": None,
        "C": float(c_value),
        "best_threshold": threshold,
        "meta_oof_accuracy": accuracy,
    }

## 5. Runtime configuration

In [6]:
RUN_DISCOVERY = True
RUN_SEARCH = True
TRAIN_FINAL_MODEL = True

MAX_SUBSET_SIZE = 3
WEIGHT_STEP_SIZE_2 = 0.025
WEIGHT_STEP_SIZE_3 = 0.05
THRESHOLD_GRID = np.round(np.arange(0.35, 0.651, 0.01), 3)
LOGREG_C_VALUES = [0.10, 0.25, 0.50, 1.0, 2.0, 4.0, 8.0, 16.0]

print(
    {
        "python": platform.python_version(),
        "sklearn": sklearn.__version__,
        "max_subset_size": MAX_SUBSET_SIZE,
        "weight_step_size_2": WEIGHT_STEP_SIZE_2,
        "weight_step_size_3": WEIGHT_STEP_SIZE_3,
        "threshold_count": int(len(THRESHOLD_GRID)),
        "logreg_C_values": LOGREG_C_VALUES,
    }
)

{'python': '3.13.0', 'sklearn': '1.5.2', 'max_subset_size': 3, 'weight_step_size_2': 0.025, 'weight_step_size_3': 0.05, 'threshold_count': 31, 'logreg_C_values': [0.1, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0]}


## 6. Discover uploaded artifacts and compute diagnostics

In [7]:
records = discover_probability_artifacts() if RUN_DISCOVERY else []
print("Discovered model artifacts:", [record["model_name"] for record in records])

meta_oof_df, meta_test_df = align_artifacts(records)
model_names = [column for column in meta_oof_df.columns if column not in {"id", "y_true"}]

summary_df = base_model_summary_table(records, meta_oof_df)
disagreement_df = pairwise_disagreement_table(meta_oof_df, model_names)
correlation_df = meta_oof_df[model_names].corr()

summary_path = CHECKPOINT_DIR / "base_model_summary.csv"
disagreement_path = CHECKPOINT_DIR / "pairwise_disagreement.csv"
correlation_path = CHECKPOINT_DIR / "probability_correlation.csv"

save_dataframe_atomic(summary_df, summary_path)
save_dataframe_atomic(disagreement_df, disagreement_path)
save_dataframe_atomic(correlation_df.reset_index().rename(columns={"index": "model_name"}), correlation_path)

display(summary_df)
if not disagreement_df.empty:
    display(disagreement_df)

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_df, annot=True, fmt=".3f", cmap="viridis")
plt.title("Correlation of base-model OOF probabilities")
save_current_figure("oof_probability_correlation.png")

plt.figure(figsize=(10, 5))
sns.barplot(data=summary_df, x="model_name", y="oof_accuracy")
plt.xticks(rotation=35, ha="right")
plt.title("OOF accuracy by discovered base model")
save_current_figure("base_model_oof_accuracy.png")

write_json_atomic(
    CHECKPOINT_DIR / "manifest.json",
    {
        "last_checkpoint_stage": "artifact_discovery_complete",
        "discovered_models": model_names,
        "n_models": len(model_names),
        "meta_oof_rows": int(len(meta_oof_df)),
        "meta_test_rows": int(len(meta_test_df)),
    },
)

Discovered model artifacts: ['challenge_08_knn_cleaning_colab_ultra', 'challenge_10_signal_features_colab_ultra']


,model_name,oof_accuracy,positive_rate,oof_path,test_path,summary_path,model_name_raw,best_stage
0,challenge_10_signal_features_colab_ultra,0.9403,0.5033,/Users/williamfrankmonroymamani/Documents/mem/...,/Users/williamfrankmonroymamani/Documents/mem/...,/Users/williamfrankmonroymamani/Documents/mem/...,Signal feature engineering model,stage2_cv
1,challenge_08_knn_cleaning_colab_ultra,0.8145,0.5315,/Users/williamfrankmonroymamani/Documents/mem/...,/Users/williamfrankmonroymamani/Documents/mem/...,/Users/williamfrankmonroymamani/Documents/mem/...,KNN with instance cleaning,stage3_local_cv


,left_model,right_model,disagreement_rate
0,challenge_08_knn_cleaning_colab_ultra,challenge_10_signal_features_colab_ultra,0.2086


## 7. Search final blending and meta-model candidates

In [8]:
search_results_path = CHECKPOINT_DIR / "final_stacking_search_results.csv"
search_df = read_dataframe(search_results_path)
completed_signatures = set(search_df["signature"]) if not search_df.empty else set()

candidate_specs = []
next_index = 0
subset_limit = min(MAX_SUBSET_SIZE, len(model_names))

for subset_size in range(2, subset_limit + 1):
    for subset in itertools.combinations(model_names, subset_size):
        subset = list(subset)
        step = WEIGHT_STEP_SIZE_2 if subset_size == 2 else WEIGHT_STEP_SIZE_3
        for weights in generate_weight_vectors(subset_size, step):
            signature = json.dumps(
                {
                    "meta_family": "weighted_average",
                    "base_models": subset,
                    "weights": weights,
                },
                sort_keys=True,
            )
            candidate_specs.append(
                {
                    "candidate_id": f"blend_{next_index:04d}",
                    "signature": signature,
                    "meta_family": "weighted_average",
                    "base_models": subset,
                    "weights": weights,
                    "C": None,
                }
            )
            next_index += 1

        for c_value in LOGREG_C_VALUES:
            signature = json.dumps(
                {
                    "meta_family": "logreg",
                    "base_models": subset,
                    "C": float(c_value),
                },
                sort_keys=True,
            )
            candidate_specs.append(
                {
                    "candidate_id": f"blend_{next_index:04d}",
                    "signature": signature,
                    "meta_family": "logreg",
                    "base_models": subset,
                    "weights": None,
                    "C": float(c_value),
                }
            )
            next_index += 1

print("Total stacking candidates:", len(candidate_specs))
print("Already completed:", len(completed_signatures))

if RUN_SEARCH:
    pending = [candidate for candidate in candidate_specs if candidate["signature"] not in completed_signatures]
    for candidate in pending:
        if candidate["meta_family"] == "weighted_average":
            result = evaluate_weighted_candidate(meta_oof_df, candidate["base_models"], candidate["weights"], THRESHOLD_GRID)
        else:
            result = evaluate_logreg_candidate(meta_oof_df, candidate["base_models"], candidate["C"], THRESHOLD_GRID)

        row = {
            "candidate_id": candidate["candidate_id"],
            "signature": candidate["signature"],
            "meta_family": result["meta_family"],
            "base_models_json": json.dumps(result["base_models"]),
            "weights_json": json.dumps(result["weights"]) if result["weights"] is not None else "",
            "C": result["C"],
            "best_threshold": result["best_threshold"],
            "meta_oof_accuracy": result["meta_oof_accuracy"],
        }
        search_df = pd.concat([search_df, pd.DataFrame([row])], ignore_index=True) if not search_df.empty else pd.DataFrame([row])
        search_df = search_df.sort_values(["meta_oof_accuracy"], ascending=[False]).reset_index(drop=True)
        save_dataframe_atomic(search_df, search_results_path)

search_df = read_dataframe(search_results_path)
display(search_df.head(20))

write_json_atomic(
    CHECKPOINT_DIR / "manifest.json",
    {
        "last_checkpoint_stage": "stacking_search_complete",
        "discovered_models": model_names,
        "completed_candidates": int(len(search_df)),
        "best_meta_oof_accuracy": None if search_df.empty else float(search_df["meta_oof_accuracy"].max()),
    },
)

Total stacking candidates: 49
Already completed: 0


,candidate_id,signature,meta_family,base_models_json,weights_json,C,best_threshold,meta_oof_accuracy
0,blend_0017,"{""base_models"": [""challenge_08_knn_cleaning_co...",weighted_average,"[""challenge_08_knn_cleaning_colab_ultra"", ""cha...","[0.425, 0.575]",NaN,0.49,0.9476
1,blend_0015,"{""base_models"": [""challenge_08_knn_cleaning_co...",weighted_average,"[""challenge_08_knn_cleaning_colab_ultra"", ""cha...","[0.375, 0.625]",NaN,0.52,0.9474
2,blend_0014,"{""base_models"": [""challenge_08_knn_cleaning_co...",weighted_average,"[""challenge_08_knn_cleaning_colab_ultra"", ""cha...","[0.35, 0.65]",NaN,0.54,0.9474
3,blend_0013,"{""base_models"": [""challenge_08_knn_cleaning_co...",weighted_average,"[""challenge_08_knn_cleaning_colab_ultra"", ""cha...","[0.325, 0.675]",NaN,0.58,0.9474
4,blend_0044,"{""C"": 1.0, ""base_models"": [""challenge_08_knn_c...",logreg,"[""challenge_08_knn_cleaning_colab_ultra"", ""cha...",NaN,1.00,0.65,0.9473
5,blend_0043,"{""C"": 0.5, ""base_models"": [""challenge_08_knn_c...",logreg,"[""challenge_08_knn_cleaning_colab_ultra"", ""cha...",NaN,0.50,0.65,0.9473
6,blend_0046,"{""C"": 4.0, ""base_models"": [""challenge_08_knn_c...",logreg,"[""challenge_08_knn_cleaning_colab_ultra"", ""cha...",NaN,4.00,0.65,0.9472
7,blend_0041,"{""C"": 0.1, ""base_models"": [""challenge_08_knn_c...",logreg,"[""challenge_08_knn_cleaning_colab_ultra"", ""cha...",NaN,0.10,0.64,0.9472
8,blend_0016,"{""base_models"": [""challenge_08_knn_cleaning_co...",weighted_average,"[""challenge_08_knn_cleaning_colab_ultra"", ""cha...","[0.4, 0.6]",NaN,0.51,0.9472
9,blend_0048,"{""C"": 16.0, ""base_models"": [""challenge_08_knn_...",logreg,"[""challenge_08_knn_cleaning_colab_ultra"", ""cha...",NaN,16.00,0.65,0.9472


## 8. Diagnostic plots

In [9]:
if not search_df.empty:
    plt.figure(figsize=(12, 6))
    plot_df = search_df.head(20).copy()
    sns.barplot(data=plot_df, x="candidate_id", y="meta_oof_accuracy", hue="meta_family")
    plt.xticks(rotation=75, ha="right")
    plt.title("Top stacking candidates")
    save_current_figure("top_stacking_candidates.png")

## 9. Train the final stacking solution and create the submission

In [10]:
if search_df.empty:
    raise RuntimeError("No stacking search results are available.")

best_row = search_df.iloc[0].to_dict()
best_base_models = json.loads(best_row["base_models_json"])
best_threshold = float(best_row["best_threshold"])
best_meta_family = best_row["meta_family"]

if best_meta_family == "weighted_average":
    best_weights = json.loads(best_row["weights_json"])
    meta_oof_prob_1 = weighted_probabilities(meta_oof_df, best_base_models, best_weights).astype(np.float32)
    meta_test_prob_1 = weighted_probabilities(meta_test_df, best_base_models, best_weights).astype(np.float32)
    final_meta_description = {
        "meta_family": best_meta_family,
        "weights": best_weights,
        "C": None,
    }
else:
    c_value = float(best_row["C"])
    meta_oof_prob_1 = generate_logreg_oof_probabilities(meta_oof_df, best_base_models, c_value).astype(np.float32)
    full_model = LogisticRegression(C=c_value, max_iter=2000)
    full_model.fit(meta_oof_df[best_base_models].to_numpy(dtype=np.float32), meta_oof_df["y_true"].to_numpy(dtype=np.int8))
    meta_test_prob_1 = full_model.predict_proba(meta_test_df[best_base_models].to_numpy(dtype=np.float32))[:, 1].astype(np.float32)
    final_meta_description = {
        "meta_family": best_meta_family,
        "weights": None,
        "C": c_value,
    }

y_true = meta_oof_df["y_true"].to_numpy(dtype=np.int8)
meta_oof_pred = (meta_oof_prob_1 >= best_threshold).astype(int)
meta_test_pred = (meta_test_prob_1 >= best_threshold).astype(int)

meta_oof_accuracy = float(accuracy_score(y_true, meta_oof_pred))
cm = confusion_matrix(y_true, meta_oof_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues", colorbar=False)
plt.title(f"Final stacking OOF confusion matrix - accuracy={meta_oof_accuracy:.4f}")
save_current_figure("final_stacking_confusion_matrix.png")

meta_oof_output = pd.DataFrame(
    {
        "id": meta_oof_df["id"].to_numpy(),
        "y_true": y_true,
        "prob_1": meta_oof_prob_1,
        "pred": meta_oof_pred.astype(int),
        "source_model": NOTEBOOK_SLUG,
    }
)
meta_test_output = pd.DataFrame(
    {
        "id": meta_test_df["id"].to_numpy(),
        "prob_1": meta_test_prob_1,
        "pred": meta_test_pred.astype(int),
        "source_model": NOTEBOOK_SLUG,
    }
)
submission_df = pd.DataFrame(
    {
        "id": meta_test_df["id"].to_numpy(),
        "class": meta_test_pred.astype(int),
    }
)

meta_oof_path = PERSIST_ROOT / "meta_oof_probabilities.csv"
meta_test_path = PERSIST_ROOT / "meta_test_probabilities.csv"
submission_path = SUBMISSION_DIR / "challenge_12_final_stacking_colab_submission.csv"
summary_path = PERSIST_ROOT / "summary.json"

save_dataframe_atomic(meta_oof_output, meta_oof_path)
save_dataframe_atomic(meta_test_output, meta_test_path)
submission_df.to_csv(submission_path, index=False)

summary_payload = {
    "model_name": "Final stacking model",
    "model_key": "final_stacking",
    "notebook_slug": NOTEBOOK_SLUG,
    "strategy": "focused_weighted_and_logreg_stacking",
    "discovered_base_models": model_names,
    "best_base_models": best_base_models,
    "best_threshold": best_threshold,
    "best_meta_oof_accuracy": meta_oof_accuracy,
    "best_candidate": final_meta_description,
    "meta_oof_path": str(meta_oof_path),
    "meta_test_path": str(meta_test_path),
    "submission_path": str(submission_path),
    "workspace_root": str(WORKSPACE_ROOT),
    "persist_root": str(PERSIST_ROOT),
}
write_json_atomic(summary_path, summary_payload)
write_json_atomic(
    CHECKPOINT_DIR / "manifest.json",
    {
        "last_checkpoint_stage": "final_model_complete",
        "best_base_models": best_base_models,
        "best_threshold": best_threshold,
        "best_meta_oof_accuracy": meta_oof_accuracy,
    },
)

print("Best candidate:", json.dumps(summary_payload, indent=2))
print("Submission path:", submission_path)

Best candidate: {
  "model_name": "Final stacking model",
  "model_key": "final_stacking",
  "notebook_slug": "challenge_12_final_stacking_colab",
  "strategy": "focused_weighted_and_logreg_stacking",
  "discovered_base_models": [
    "challenge_08_knn_cleaning_colab_ultra",
    "challenge_10_signal_features_colab_ultra"
  ],
  "best_base_models": [
    "challenge_08_knn_cleaning_colab_ultra",
    "challenge_10_signal_features_colab_ultra"
  ],
  "best_threshold": 0.49,
  "best_meta_oof_accuracy": 0.9476,
  "best_candidate": {
    "meta_family": "weighted_average",
    "weights": [
      0.425,
      0.575
    ],
    "C": null
  },
  "meta_oof_path": "/Users/williamfrankmonroymamani/Documents/mem/stadistics/mem-estadistics/challenge/results-pre-stack/output/challenge_12_final_stacking_colab/meta_oof_probabilities.csv",
  "meta_test_path": "/Users/williamfrankmonroymamani/Documents/mem/stadistics/mem-estadistics/challenge/results-pre-stack/output/challenge_12_final_stacking_colab/meta_t

## 10. Optional: export and download a resume bundle

In [11]:
DOWNLOAD_BUNDLE_NOW = False

bundle_path = create_resume_bundle()
print("Resume bundle saved to:", bundle_path)

if DOWNLOAD_BUNDLE_NOW and IN_COLAB:
    files.download(str(bundle_path))

Resume bundle saved to: /Users/williamfrankmonroymamani/Documents/mem/stadistics/mem-estadistics/challenge/results-pre-stack/exports/challenge_12_final_stacking_colab_resume.zip


## Notes

Flujo recomendado para tu caso actual:

1. empaquetar localmente `knn_cleaning_ultra` y `signal_features_ultra`
2. subir esos ZIPs a esta notebook
3. correr el stacking final
4. si luego completas `svm_preprocessing`, subir su ZIP y repetir

Esta notebook no necesita los CSV originales del challenge.